# Airline Flight Delay Analytics & Prediction

**Big Data Analytics (BDA) project — B.E. CSE, SFIT Mumbai**

This notebook is a Colab-runnable, narrated consolidation of the project's full
PySpark pipeline: it ingests and joins the raw Kaggle `usdot/flight-delays`
CSVs (2015 US domestic flights, ~5.8M rows) using an explicit schema, cleans
the data and builds a FAA-standard `is_delayed` label, aggregates delay rates
across airline / airport / hour / day-of-week / month, visualizes the
findings, and trains a class-imbalance-aware Random Forest classifier to
predict delays from information known before departure. Every section mirrors
the equivalent verified script under `src/` (`ingest.py` → `clean.py` →
`aggregate.py` → `eda.py` → `model.py`) — this notebook adapts paths for
Colab's `/content/` filesystem and adds narration, but does not diverge from
the pipeline logic already validated locally.

## 1. Setup

This notebook is designed to run top-to-bottom on **Google Colab**.

**Colab vs. local setup:** Colab's runtime already ships a Java Runtime
Environment, which PySpark needs under the hood to launch its JVM gateway —
so unlike the local development environment for this project (which required
installing `openjdk@17` via Homebrew on macOS before PySpark would run at
all), no Java installation step is needed here. We only need to install
PySpark itself.

In [ ]:
!pip install -q pyspark

## 2. Download the dataset (Kaggle `usdot/flight-delays`)

This notebook needs your Kaggle API credentials to download the dataset.
Two ways to provide them:

1. **Upload `kaggle.json` directly** — run the upload cell below and use the
   file picker. You can generate `kaggle.json` at
   https://www.kaggle.com/settings → API → "Create New Token".
2. **Colab Secrets manager** — instead of uploading a file, store your
   Kaggle username/key as Colab secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) via
   the 🔑 icon in the left sidebar, then skip the upload cell and use the
   commented `userdata` alternative in the cell after it.

In [ ]:
from google.colab import files

# Upload the kaggle.json file you downloaded from https://www.kaggle.com/settings
# (skip this cell if you're using Colab Secrets instead — see markdown above)
uploaded = files.upload()

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Alternative if using Colab Secrets instead of the file-upload cell above:
# from google.colab import userdata
# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!pip install -q kaggle
!mkdir -p /content/data/raw
!kaggle datasets download -d usdot/flight-delays -p /content/data/raw --unzip
!ls -la /content/data/raw

## 3. Ingest — explicit schema, join lookup tables

`flights.csv` is ~5.8M rows. Letting Spark run `inferSchema` on a file this
size means an extra full pass just to guess column types before the real
read even starts — wasteful at this scale, so instead we define an explicit
`StructType` up front (this mirrors `src/ingest.py` and the project's own
"Do NOT run schema inference on the full 5.8M row CSV" rule).

We then left-join:
- `flights.AIRLINE` (IATA code) → `airlines.csv` → full airline name
- `flights.ORIGIN_AIRPORT` (IATA code) → `airports.csv` → origin city/state

The result is a wide, **joined-but-unfiltered** dataframe — no rows are
dropped yet, that happens in the cleaning step next.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
)

RAW_DIR = "/content/data/raw"
PROCESSED_DIR = "/content/data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

FLIGHTS_SCHEMA = StructType(
    [
        StructField("YEAR", IntegerType(), True),
        StructField("MONTH", IntegerType(), True),
        StructField("DAY", IntegerType(), True),
        StructField("DAY_OF_WEEK", IntegerType(), True),
        StructField("AIRLINE", StringType(), True),
        StructField("FLIGHT_NUMBER", IntegerType(), True),
        StructField("TAIL_NUMBER", StringType(), True),
        StructField("ORIGIN_AIRPORT", StringType(), True),
        StructField("DESTINATION_AIRPORT", StringType(), True),
        StructField("SCHEDULED_DEPARTURE", IntegerType(), True),
        StructField("DEPARTURE_TIME", IntegerType(), True),
        StructField("DEPARTURE_DELAY", DoubleType(), True),
        StructField("TAXI_OUT", DoubleType(), True),
        StructField("WHEELS_OFF", IntegerType(), True),
        StructField("SCHEDULED_TIME", DoubleType(), True),
        StructField("ELAPSED_TIME", DoubleType(), True),
        StructField("AIR_TIME", DoubleType(), True),
        StructField("DISTANCE", IntegerType(), True),
        StructField("WHEELS_ON", IntegerType(), True),
        StructField("TAXI_IN", DoubleType(), True),
        StructField("SCHEDULED_ARRIVAL", IntegerType(), True),
        StructField("ARRIVAL_TIME", IntegerType(), True),
        StructField("ARRIVAL_DELAY", DoubleType(), True),
        StructField("DIVERTED", IntegerType(), True),
        StructField("CANCELLED", IntegerType(), True),
        StructField("CANCELLATION_REASON", StringType(), True),
        StructField("AIR_SYSTEM_DELAY", DoubleType(), True),
        StructField("SECURITY_DELAY", DoubleType(), True),
        StructField("AIRLINE_DELAY", DoubleType(), True),
        StructField("LATE_AIRCRAFT_DELAY", DoubleType(), True),
        StructField("WEATHER_DELAY", DoubleType(), True),
    ]
)

AIRLINES_SCHEMA = StructType(
    [
        StructField("IATA_CODE", StringType(), True),
        StructField("AIRLINE", StringType(), True),
    ]
)

AIRPORTS_SCHEMA = StructType(
    [
        StructField("IATA_CODE", StringType(), True),
        StructField("AIRPORT", StringType(), True),
        StructField("CITY", StringType(), True),
        StructField("STATE", StringType(), True),
        StructField("COUNTRY", StringType(), True),
        StructField("LATITUDE", DoubleType(), True),
        StructField("LONGITUDE", DoubleType(), True),
    ]
)

spark = (
    SparkSession.builder.appName("AirlineDelayAnalytics-Ingest")
    .master("local[*]")
    # 4g driver memory: the aggregation and modeling steps later in this
    # notebook shuffle/persist enough data that the default 1g driver heap
    # runs out (this one SparkSession is reused for the whole notebook).
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

flights = spark.read.csv(f"{RAW_DIR}/flights.csv", header=True, schema=FLIGHTS_SCHEMA)
airlines = spark.read.csv(f"{RAW_DIR}/airlines.csv", header=True, schema=AIRLINES_SCHEMA)
airports = spark.read.csv(f"{RAW_DIR}/airports.csv", header=True, schema=AIRPORTS_SCHEMA)

# Join airline full name (rename to avoid colliding with flights.AIRLINE code column)
airlines_renamed = airlines.withColumnRenamed("AIRLINE", "AIRLINE_NAME")
joined = flights.join(
    airlines_renamed, flights["AIRLINE"] == airlines_renamed["IATA_CODE"], "left"
).drop(airlines_renamed["IATA_CODE"])

# Join origin airport city/state
origin_airports = (
    airports.withColumnRenamed("CITY", "ORIGIN_CITY")
    .withColumnRenamed("STATE", "ORIGIN_STATE")
    .select("IATA_CODE", "ORIGIN_CITY", "ORIGIN_STATE")
)
joined = joined.join(
    origin_airports, joined["ORIGIN_AIRPORT"] == origin_airports["IATA_CODE"], "left"
).drop(origin_airports["IATA_CODE"])

row_count = joined.count()
print(f"\nJoined row count: {row_count}\n")
print("Joined schema:")
joined.printSchema()

joined.write.mode("overwrite").parquet(f"{PROCESSED_DIR}/flights_joined.parquet")

print("\nSample rows:")
joined.select(
    "YEAR", "MONTH", "DAY", "AIRLINE", "AIRLINE_NAME",
    "ORIGIN_AIRPORT", "ORIGIN_CITY", "ORIGIN_STATE",
    "DESTINATION_AIRPORT", "ARRIVAL_DELAY",
).show(5, truncate=False)

## 4. Clean — filter invalid flights, build the `is_delayed` label

Two things need to happen before this data is model-ready:

1. **Drop cancelled/diverted flights.** A cancelled or diverted flight has no
   meaningful `ARRIVAL_DELAY` — there's nothing to predict — so we filter
   `CANCELLED == 1` and `DIVERTED == 1` out and print exactly how many rows
   each accounts for, so the data loss is visible and explainable.
2. **Build the label.** We use the FAA's standard definition of a delayed
   flight: `is_delayed = 1` if `ARRIVAL_DELAY >= 15` minutes, else `0`. This
   is a widely-used, well-understood threshold rather than an arbitrary cutoff.

We also fill nulls in the five `*_DELAY` cause columns with `0` — a null
there means "no delay of that type occurred," not missing data, so filling
with 0 is correct rather than dropping those rows. Finally we derive
`SCHEDULED_DEPARTURE_HOUR` from the `SCHEDULED_DEPARTURE` HHMM integer field
(e.g. `1430` → hour `14`), used later as a model feature and EDA dimension.

In [ ]:
from pyspark.sql import functions as F

DELAY_CAUSE_COLUMNS = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
]

df = spark.read.parquet(f"{PROCESSED_DIR}/flights_joined.parquet")

total_rows = df.count()
print(f"\nStarting row count: {total_rows}")

cancelled_count = df.filter(F.col("CANCELLED") == 1).count()
diverted_count = df.filter((F.col("CANCELLED") != 1) & (F.col("DIVERTED") == 1)).count()
print(f"Rows dropped for CANCELLED == 1: {cancelled_count}")
print(f"Rows dropped for DIVERTED == 1 (excluding already-cancelled): {diverted_count}")

df = df.filter((F.col("CANCELLED") != 1) & (F.col("DIVERTED") != 1))
after_filter_rows = df.count()
print(f"Row count after dropping cancelled/diverted: {after_filter_rows}")

df = df.fillna(0, subset=DELAY_CAUSE_COLUMNS)

df = df.withColumn("is_delayed", F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0))

delayed_count = df.filter(F.col("is_delayed") == 1).count()
on_time_count = after_filter_rows - delayed_count
pct_delayed = 100.0 * delayed_count / after_filter_rows
pct_on_time = 100.0 * on_time_count / after_filter_rows
print(
    f"\nClass balance — delayed: {delayed_count} ({pct_delayed:.2f}%), "
    f"on-time: {on_time_count} ({pct_on_time:.2f}%)"
)

df = df.withColumn(
    "SCHEDULED_DEPARTURE_HOUR", (F.col("SCHEDULED_DEPARTURE") / 100).cast("int")
)

final_row_count = df.count()
print(f"\nFinal row count: {final_row_count}")

df.write.mode("overwrite").parquet(f"{PROCESSED_DIR}/flights_clean.parquet")

print("\nSample rows:")
df.select(
    "YEAR", "MONTH", "DAY", "AIRLINE", "SCHEDULED_DEPARTURE",
    "SCHEDULED_DEPARTURE_HOUR", "ARRIVAL_DELAY", "is_delayed",
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",
).show(5, truncate=False)

## 5. Aggregate — delay rates by airline, airport, hour, day-of-week, month

For each dimension we compute `total_flights`, `total_delayed`, and
`delay_pct`. Results are saved as both Parquet (for downstream use) and CSV
(for quick eyeballing / the EDA step below).

> **Data quality finding worth calling out:** while building the by-airport
> aggregation, we discovered that **all of October 2015** (~8.4% of rows,
> 482,878 flights) uses **numeric DOT airport IDs instead of 3-letter IATA
> codes** in `ORIGIN_AIRPORT`/`DESTINATION_AIRPORT` — a known quirk of the
> Kaggle `usdot/flight-delays` dataset. Those rows never matched
> `airports.csv` during the ingest join (their `ORIGIN_CITY`/`ORIGIN_STATE`
> are null), and if left in, they'd surface as junk numeric "airports" in
> any airport-keyed table or chart. The fix: filter `ORIGIN_AIRPORT` to
> `^[A-Z]{3}$` **only for the airport-keyed aggregation** — every other
> aggregation (airline/hour/day/month) and the model training data are
> unaffected, since `is_delayed` itself only depends on `ARRIVAL_DELAY`, not
> on whether the airport code parsed as IATA.

In [ ]:
import json

from pyspark import StorageLevel

METRICS_DIR = "/content/outputs/metrics"
os.makedirs(METRICS_DIR, exist_ok=True)

DELAY_CAUSE_COLS_AGG = [
    "AIRLINE_DELAY", "WEATHER_DELAY", "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
]


def delay_rate_agg(frame, group_cols):
    return (
        frame.groupBy(*group_cols)
        .agg(F.count("*").alias("total_flights"), F.sum("is_delayed").alias("total_delayed"))
        .withColumn("delay_pct", F.round(100.0 * F.col("total_delayed") / F.col("total_flights"), 2))
    )


def save(frame, name):
    frame.coalesce(1).write.mode("overwrite").parquet(f"{PROCESSED_DIR}/{name}.parquet")
    frame.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{PROCESSED_DIR}/{name}.csv")


df.persist(StorageLevel.DISK_ONLY)

# a. By AIRLINE_NAME
by_airline = delay_rate_agg(df, ["AIRLINE_NAME"]).orderBy(F.desc("delay_pct"))
save(by_airline, "agg_by_airline")

# b. By ORIGIN_AIRPORT, IATA-only + busy airports (see data quality note above)
IATA_CODE_PATTERN = "^[A-Z]{3}$"
airport_rows_before = df.select("ORIGIN_AIRPORT").count()
valid_airport_df = df.filter(F.col("ORIGIN_AIRPORT").rlike(IATA_CODE_PATTERN))
airport_rows_after = valid_airport_df.count()
print(
    f"By-airport aggregation: excluded {airport_rows_before - airport_rows_after} rows "
    f"with non-IATA ORIGIN_AIRPORT codes (kept {airport_rows_after} of {airport_rows_before})"
)

by_airport = (
    delay_rate_agg(valid_airport_df, ["ORIGIN_AIRPORT"])
    .filter(F.col("total_flights") > 10000)
    .orderBy(F.desc("total_flights"))
)
save(by_airport, "agg_by_airport")

# c. By SCHEDULED_DEPARTURE_HOUR
by_hour = delay_rate_agg(df, ["SCHEDULED_DEPARTURE_HOUR"]).orderBy(F.asc("SCHEDULED_DEPARTURE_HOUR"))
save(by_hour, "agg_by_hour")

# d. By DAY_OF_WEEK
by_dow = delay_rate_agg(df, ["DAY_OF_WEEK"]).orderBy(F.asc("DAY_OF_WEEK"))
save(by_dow, "agg_by_day_of_week")

# e. By MONTH
by_month = delay_rate_agg(df, ["MONTH"]).orderBy(F.asc("MONTH"))
save(by_month, "agg_by_month")

# Overall delay-cause totals (minutes) -> pie chart in the EDA section
cause_sums = df.agg(*[F.sum(c).alias(c) for c in DELAY_CAUSE_COLS_AGG]).collect()[0]
cause_totals = {c: float(cause_sums[c] or 0.0) for c in DELAY_CAUSE_COLS_AGG}
with open(f"{METRICS_DIR}/delay_cause_totals.json", "w") as f:
    json.dump(cause_totals, f, indent=2)

print("\n(a) Delay rate by airline (top 10):")
by_airline.show(10, truncate=False)
print("(b) Delay rate by origin airport, total_flights > 10000 (top 10 busiest):")
by_airport.show(10, truncate=False)
print("(c) Delay rate by scheduled departure hour (top 10):")
by_hour.show(10, truncate=False)
print("(d) Delay rate by day of week:")
by_dow.show(10, truncate=False)
print("(e) Delay rate by month (top 10):")
by_month.show(10, truncate=False)
print("Overall delay-cause totals (minutes):")
print(json.dumps(cause_totals, indent=2))

total_rows_agg = df.count()
overall_delay_pct = round(100.0 * df.filter(F.col("is_delayed") == 1).count() / total_rows_agg, 2)
print(f"\nOverall delay percentage: {overall_delay_pct}% (sanity check vs 18.61% from the clean step)")

## 6. Exploratory Data Analysis — 6 charts

We render the six charts inline (via `%matplotlib inline`) directly from the
CSV outputs of the aggregation step above, and also save them to
`/content/outputs/charts/` as PNGs — matching `src/eda.py` exactly, just with
`plt.show()` added so they display in the notebook output too.

In [ ]:
%matplotlib inline
import glob

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

CHARTS_DIR = "/content/outputs/charts"
os.makedirs(CHARTS_DIR, exist_ok=True)
sns.set_style("whitegrid")
DPI = 150

MONTH_LABELS = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
                7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
DOW_LABELS = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}


def read_agg_csv(name):
    path = glob.glob(f"{PROCESSED_DIR}/{name}.csv/part-*.csv")[0]
    return pd.read_csv(path)


findings = []

# 1. Top 10 worst airlines by delay_pct
by_airline_pd = read_agg_csv("agg_by_airline").sort_values("delay_pct", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_airline_pd.sort_values("delay_pct"), x="delay_pct", y="AIRLINE_NAME", color="#4C72B0", ax=ax)
ax.set_title("Top 10 Worst Airlines by Delay Rate")
ax.set_xlabel("Delay Rate (%)")
ax.set_ylabel("Airline")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/01_delay_by_airline.png", dpi=DPI)
plt.show()

worst = by_airline_pd.iloc[0]
best_of_worst = by_airline_pd.iloc[-1]
findings.append(
    f"Chart 1 (delay by airline): {worst['AIRLINE_NAME']} has the worst delay rate at "
    f"{worst['delay_pct']}%, roughly {worst['delay_pct'] / best_of_worst['delay_pct']:.1f}x the "
    f"rate of {best_of_worst['AIRLINE_NAME']} ({best_of_worst['delay_pct']}%), the "
    f"best-performing airline within this worst-10 list."
)

# 2. Delay pct for top 10 busiest airports
by_airport_pd = read_agg_csv("agg_by_airport").sort_values("total_flights", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_airport_pd.sort_values("total_flights"), x="delay_pct", y="ORIGIN_AIRPORT", color="#DD8452", ax=ax)
ax.set_title("Delay Rate at the 10 Busiest Origin Airports")
ax.set_xlabel("Delay Rate (%)")
ax.set_ylabel("Airport (IATA code)")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/02_delay_by_airport.png", dpi=DPI)
plt.show()

worst_airport = by_airport_pd.loc[by_airport_pd["delay_pct"].idxmax()]
best_airport = by_airport_pd.loc[by_airport_pd["delay_pct"].idxmin()]
findings.append(
    f"Chart 2 (delay by airport): among the 10 busiest airports, {worst_airport['ORIGIN_AIRPORT']} "
    f"has the highest delay rate ({worst_airport['delay_pct']}%) while {best_airport['ORIGIN_AIRPORT']} "
    f"has the lowest ({best_airport['delay_pct']}%) despite similar traffic volumes — busiest does "
    f"not mean most delayed."
)

# 3. Delay cause breakdown pie chart
with open(f"{METRICS_DIR}/delay_cause_totals.json") as f:
    cause_totals = json.load(f)
labels = list(cause_totals.keys())
values = list(cause_totals.values())
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(values, labels=labels, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("Set2", len(values)))
ax.set_title("Total Delay Minutes by Cause")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/03_delay_cause_breakdown.png", dpi=DPI)
plt.show()

total_minutes = sum(values)
top_cause = max(cause_totals, key=cause_totals.get)
top_cause_pct = 100.0 * cause_totals[top_cause] / total_minutes
findings.append(
    f"Chart 3 (delay cause breakdown): {top_cause.replace('_', ' ').title()} is the single largest "
    f"contributor to total delay minutes, accounting for {top_cause_pct:.1f}% of all delay time."
)

# 4. Delay pct across scheduled departure hour
by_hour_pd = read_agg_csv("agg_by_hour").sort_values("SCHEDULED_DEPARTURE_HOUR")
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=by_hour_pd, x="SCHEDULED_DEPARTURE_HOUR", y="delay_pct", marker="o", ax=ax)
ax.set_title("Delay Rate by Scheduled Departure Hour")
ax.set_xlabel("Scheduled Departure Hour (0-23)")
ax.set_ylabel("Delay Rate (%)")
ax.set_xticks(range(0, 24))
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/04_delay_by_hour.png", dpi=DPI)
plt.show()

morning = by_hour_pd[by_hour_pd["SCHEDULED_DEPARTURE_HOUR"].between(5, 8)]["delay_pct"].mean()
evening = by_hour_pd[by_hour_pd["SCHEDULED_DEPARTURE_HOUR"].between(18, 21)]["delay_pct"].mean()
findings.append(
    f"Chart 4 (delay by hour): delay rate climbs steadily through the day — early-morning flights "
    f"(5-8am) average {morning:.1f}% delayed versus {evening:.1f}% for evening flights (6-9pm), "
    f"roughly {evening / morning:.1f}x higher."
)

# 5. Delay pct by day of week
by_dow_pd = read_agg_csv("agg_by_day_of_week").sort_values("DAY_OF_WEEK")
by_dow_pd["DOW_LABEL"] = by_dow_pd["DAY_OF_WEEK"].map(DOW_LABELS)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_dow_pd, x="DOW_LABEL", y="delay_pct", color="#55A868", ax=ax)
ax.set_title("Delay Rate by Day of Week")
ax.set_xlabel("Day of Week")
ax.set_ylabel("Delay Rate (%)")
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/05_delay_by_dayofweek.png", dpi=DPI)
plt.show()

worst_dow = by_dow_pd.loc[by_dow_pd["delay_pct"].idxmax()]
best_dow = by_dow_pd.loc[by_dow_pd["delay_pct"].idxmin()]
findings.append(
    f"Chart 5 (delay by day of week): {worst_dow['DOW_LABEL']} is the worst day to fly "
    f"({worst_dow['delay_pct']}% delayed), while {best_dow['DOW_LABEL']} is the best "
    f"({best_dow['delay_pct']}%)."
)

# 6. Delay pct by month
by_month_pd = read_agg_csv("agg_by_month").sort_values("MONTH")
by_month_pd["MONTH_LABEL"] = by_month_pd["MONTH"].map(MONTH_LABELS)
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=by_month_pd, x="MONTH", y="delay_pct", marker="o", ax=ax)
ax.set_title("Delay Rate by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Delay Rate (%)")
ax.set_xticks(by_month_pd["MONTH"])
ax.set_xticklabels(by_month_pd["MONTH_LABEL"])
fig.tight_layout()
fig.savefig(f"{CHARTS_DIR}/06_delay_by_month.png", dpi=DPI)
plt.show()

worst_month = by_month_pd.loc[by_month_pd["delay_pct"].idxmax()]
best_month = by_month_pd.loc[by_month_pd["delay_pct"].idxmin()]
findings.append(
    f"Chart 6 (delay by month): {worst_month['MONTH_LABEL']} has the highest delay rate "
    f"({worst_month['delay_pct']}%), while {best_month['MONTH_LABEL']} has the lowest "
    f"({best_month['delay_pct']}%)."
)

print("=== Findings summary ===")
for line in findings:
    print(f"- {line}")

## 7. Model — predicting delays from pre-flight information

**Features used (all known before departure — no data leakage):** `AIRLINE`
and `ORIGIN_AIRPORT` (indexed), `MONTH`, `DAY_OF_WEEK`,
`SCHEDULED_DEPARTURE_HOUR`, `DISTANCE`. We deliberately exclude every
post-flight column — `ARRIVAL_DELAY`, `DEPARTURE_DELAY`, `ARRIVAL_TIME`,
`TAXI_IN`/`TAXI_OUT`, `AIR_TIME`, `WHEELS_ON`/`WHEELS_OFF`, and the `*_DELAY`
minute columns — since a model trained on those would just be cheating by
looking at outcomes only known after the flight already happened.

**The class imbalance problem:** only 18.61% of flights are delayed. A model
that predicts "on-time" for every single flight scores ~81% accuracy without
learning anything at all. We first train plain (unweighted) Logistic
Regression and Random Forest models to show this failure mode concretely,
then fix it with inverse-frequency class weighting plus a threshold sweep
tuned for the delayed class's F1 score.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array

LABEL_COL = "is_delayed"
FEATURE_COLS = [
    "AIRLINE_INDEXED", "ORIGIN_AIRPORT_INDEXED", "MONTH",
    "DAY_OF_WEEK", "SCHEDULED_DEPARTURE_HOUR", "DISTANCE",
]

model_df = spark.read.parquet(f"{PROCESSED_DIR}/flights_clean.parquet").select(
    "AIRLINE", "ORIGIN_AIRPORT", "MONTH", "DAY_OF_WEEK",
    "SCHEDULED_DEPARTURE_HOUR", "DISTANCE", LABEL_COL,
)
model_df.persist(StorageLevel.DISK_ONLY)
print(f"Total rows for modeling: {model_df.count()}")

airline_indexer = StringIndexer(inputCol="AIRLINE", outputCol="AIRLINE_INDEXED", handleInvalid="keep")
airport_indexer = StringIndexer(inputCol="ORIGIN_AIRPORT", outputCol="ORIGIN_AIRPORT_INDEXED", handleInvalid="keep")
model_df = airline_indexer.fit(model_df).transform(model_df)
model_df = airport_indexer.fit(model_df).transform(model_df)

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features")
model_df = assembler.transform(model_df)

train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
train_df.persist(StorageLevel.DISK_ONLY)
test_df.persist(StorageLevel.DISK_ONLY)
print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")

test_on_time = test_df.filter(f"{LABEL_COL} == 0").count()
majority_baseline_accuracy = round(test_on_time / test_df.count(), 4)
print(f"\n'Always predict on-time' baseline accuracy on test set: {majority_baseline_accuracy}")


def evaluate(predictions, model_name):
    auc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    ).evaluate(predictions)
    metrics = {
        m: MulticlassClassificationEvaluator(labelCol=LABEL_COL, predictionCol="prediction", metricName=m).evaluate(predictions)
        for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]
    }
    confusion = predictions.groupBy(LABEL_COL, "prediction").count().orderBy(LABEL_COL, "prediction").collect()
    confusion_counts = {f"actual_{int(r[LABEL_COL])}_predicted_{int(r['prediction'])}": r["count"] for r in confusion}
    result = {
        "model": model_name, "auc": round(auc, 4), "accuracy": round(metrics["accuracy"], 4),
        "f1": round(metrics["f1"], 4), "precision": round(metrics["weightedPrecision"], 4),
        "recall": round(metrics["weightedRecall"], 4), "confusion_matrix": confusion_counts,
    }
    print(f"\n=== {model_name} ===")
    for k in ["auc", "accuracy", "f1", "precision", "recall"]:
        print(f"{k}: {result[k]}")
    print(f"Confusion matrix: {confusion_counts}")
    return result


# --- Baseline (unweighted) models — expected to degenerate to majority-class ---
lr = LogisticRegression(featuresCol="features", labelCol=LABEL_COL)
lr_result = evaluate(lr.fit(train_df).transform(test_df), "LogisticRegression (unweighted)")

rf = RandomForestClassifier(
    featuresCol="features", labelCol=LABEL_COL, numTrees=50, maxDepth=8, maxBins=700, seed=42,
)  # maxBins must be >= ORIGIN_AIRPORT's ~628 distinct indexed values
rf_model = rf.fit(train_df)
rf_result = evaluate(rf_model.transform(test_df), "RandomForest (unweighted)")

print(
    "\n-> As expected: both unweighted models are functionally identical to the "
    f"majority-class baseline ({majority_baseline_accuracy * 100:.2f}% accuracy) — "
    "0 true positives for the delayed class."
)

### Fixing the imbalance: class weighting + threshold tuning

We compute inverse-frequency class weights
(`weight = total_count / (2 * class_count)`), attach them as a training
column, and retrain both models with `weightCol`. Weighting alone shifts the
probability distribution but doesn't guarantee the default 0.5 threshold is
still the right cutoff — so we sweep thresholds from 0.30 to 0.60 and report
precision/recall/F1 for the delayed class specifically at each one (accuracy
alone would be misleading here, since the majority-class baseline already
gets ~81% "for free").

In [ ]:
THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]


def sweep_thresholds(predictions, model_name):
    predictions = predictions.withColumn("prob_delayed", vector_to_array("probability")[1])
    predictions.persist(StorageLevel.DISK_ONLY)
    predictions.count()

    print(f"\n{model_name} — threshold sweep (precision/recall/F1 for delayed class):")
    print(f"{'threshold':>10} {'precision':>10} {'recall':>10} {'f1':>10}")

    sweep_results = []
    for t in THRESHOLDS:
        row = predictions.agg(
            F.sum(F.when((F.col(LABEL_COL) == 1) & (F.col("prob_delayed") >= t), 1).otherwise(0)).alias("tp"),
            F.sum(F.when((F.col(LABEL_COL) == 0) & (F.col("prob_delayed") >= t), 1).otherwise(0)).alias("fp"),
            F.sum(F.when((F.col(LABEL_COL) == 1) & (F.col("prob_delayed") < t), 1).otherwise(0)).alias("fn"),
            F.sum(F.when((F.col(LABEL_COL) == 0) & (F.col("prob_delayed") < t), 1).otherwise(0)).alias("tn"),
        ).collect()[0]
        tp, fp, fn = row["tp"], row["fp"], row["fn"]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        sweep_results.append({"threshold": t, "precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)})
        print(f"{t:>10.2f} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f}")

    predictions.unpersist()
    best = max(sweep_results, key=lambda r: r["f1"])
    print(f"Best threshold for {model_name} (max F1): {best['threshold']} -> {best}")
    return sweep_results, best


label_counts = {int(r[LABEL_COL]): r["count"] for r in train_df.groupBy(LABEL_COL).count().collect()}
total_train = sum(label_counts.values())
class_weights = {label: total_train / (2 * count) for label, count in label_counts.items()}
print(f"Class weights (total / (2 * class_count)): {class_weights}")

train_weighted = train_df.withColumn(
    "classWeight", F.when(F.col(LABEL_COL) == 1, class_weights[1]).otherwise(class_weights[0])
)
train_weighted.persist(StorageLevel.DISK_ONLY)
train_weighted.count()

lr_weighted = LogisticRegression(featuresCol="features", labelCol=LABEL_COL, weightCol="classWeight")
lr_weighted_predictions = lr_weighted.fit(train_weighted).transform(test_df)
print(f"\nWeighted LogisticRegression AUC: {round(BinaryClassificationEvaluator(labelCol=LABEL_COL).evaluate(lr_weighted_predictions), 4)}")
lr_sweep, lr_best = sweep_thresholds(lr_weighted_predictions, "Weighted LogisticRegression")

rf_weighted = RandomForestClassifier(
    featuresCol="features", labelCol=LABEL_COL, numTrees=50, maxDepth=8,
    maxBins=700, seed=42, weightCol="classWeight",
)
rf_weighted_model = rf_weighted.fit(train_weighted)
rf_weighted_predictions = rf_weighted_model.transform(test_df)
rf_weighted_auc = BinaryClassificationEvaluator(labelCol=LABEL_COL).evaluate(rf_weighted_predictions)
print(f"\nWeighted RandomForest AUC: {round(rf_weighted_auc, 4)}")
rf_sweep, rf_best = sweep_thresholds(rf_weighted_predictions, "Weighted RandomForest")

# Confusion matrix for the weighted RF model at its best threshold
rf_best_t = rf_best["threshold"]
rf_tuned = rf_weighted_predictions.withColumn(
    "prob_delayed", vector_to_array("probability")[1]
).withColumn("tuned_prediction", F.when(F.col("prob_delayed") >= rf_best_t, 1).otherwise(0))
rf_tuned_confusion = rf_tuned.groupBy(LABEL_COL, "tuned_prediction").count().orderBy(LABEL_COL, "tuned_prediction").collect()
rf_tuned_confusion_counts = {f"actual_{int(r[LABEL_COL])}_predicted_{int(r['tuned_prediction'])}": r["count"] for r in rf_tuned_confusion}

print("\n=== Before vs After ===")
print(f"BEFORE (unweighted RF, 0.5 threshold): {rf_result['confusion_matrix']} -> 0 true positives, identical to majority baseline")
print(f"AFTER  (weighted RF, tuned threshold={rf_best_t}): precision={rf_best['precision']}, recall={rf_best['recall']}, f1={rf_best['f1']}, confusion={rf_tuned_confusion_counts}")
print(
    "\nHonest trade-off: the weighted model now catches a meaningful share of real delays "
    "(non-zero recall), at the cost of many false alarms (precision well below 1.0) — this is "
    "the expected precision/recall trade-off from inverse-frequency class weighting, not a free win."
)

feature_importances = sorted(zip(FEATURE_COLS, rf_weighted_model.featureImportances.toArray()), key=lambda x: x[1], reverse=True)
print("\nWeighted RandomForest feature importances (sorted descending):")
for feature, importance in feature_importances:
    print(f"  {feature}: {round(float(importance), 4)}")

model_metrics = {
    "majority_class_baseline_accuracy": majority_baseline_accuracy,
    "unweighted_logistic_regression": lr_result,
    "unweighted_random_forest": rf_result,
    "weighted_logistic_regression": {"auc": round(BinaryClassificationEvaluator(labelCol=LABEL_COL).evaluate(lr_weighted_predictions), 4), "threshold_sweep": lr_sweep, "best_threshold": lr_best},
    "weighted_random_forest": {"auc": round(rf_weighted_auc, 4), "threshold_sweep": rf_sweep, "best_threshold": rf_best, "confusion_matrix_at_best_threshold": rf_tuned_confusion_counts},
}
with open(f"{METRICS_DIR}/model_metrics_notebook.json", "w") as f:
    json.dump(model_metrics, f, indent=2)
print(f"\nMetrics saved to {METRICS_DIR}/model_metrics_notebook.json")

## 8. Conclusion

**Key findings from the EDA (5.7M cleaned flights, 18.61% delayed overall):**

- **Worst airline:** Spirit Air Lines, 29.71% delay rate — roughly 1.6x the
  rate of the best airline among the worst-10.
- **Worst busy airport:** ORD (Chicago O'Hare), 24.1% among the top-10
  busiest airports — while ATL, the single busiest airport, has a
  comparatively low 16.44%. Busiest does not mean most delayed.
- **Top delay cause:** Late Aircraft Delay, accounting for ~39.8% of all
  delay minutes — more than double Weather Delay's share, i.e. most delay
  time comes from operational knock-on effects, not weather.
- **Time-of-day pattern:** delay rate climbs steadily through the day —
  early-morning flights (5-8am) average roughly 10% delayed versus ~26% for
  evening flights (6-9pm), consistent with delays cascading as aircraft fall
  further behind schedule.
- **Day-of-week pattern:** Thursday is the worst day to fly (~20.0%),
  Saturday the best (~16.0%).
- **Month pattern:** June is the worst month (~23.5%), October the best
  (~12.4%) among the months studied.
- **Data quality:** ~8.4% of rows (all of October 2015) use numeric DOT
  airport IDs instead of IATA codes — filtered out of airport-specific views
  only, since it doesn't affect the delay label itself.

**Model performance, honestly reported:** with only six pre-flight-known
features and no imbalance handling, both Logistic Regression and Random
Forest degenerate to the majority-class baseline (~81% accuracy, 0 true
positives for the delayed class) — a model that never predicts "delayed" is
not useful, even though it looks accurate. Applying inverse-frequency class
weighting plus a threshold sweep tuned for the delayed class's F1 score
produces a materially different, and more honest, trade-off: the tuned
weighted Random Forest (threshold ≈ 0.5) catches **~64% of real delays
(recall)**, but only **~25-26% of its "delayed" predictions are correct
(precision)** — roughly three out of four delay alarms are false positives.
This is the expected cost of correcting for severe class imbalance with only
a handful of coarse features (airline, airport, month, day-of-week,
scheduled hour, distance); it is a usable *screening* signal, not a
high-confidence prediction, and any downstream use (e.g. the project's
dashboard) should present it with that caveat rather than as a certainty.